### Set up Env

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

```bash
# # another way of running vllm without creating any permanent environment
# uv run --with vllm vllm --help
```

```bash
export UV_TORCH_BACKEND=auto
uv sync
source .venv/bin/activate
```

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

### Offline Batched Inference

In [ ]:
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]

sampling_params = SamplingParams(
    temperature=0.8,  # Higher for more creativity
    top_p=0.95,  # Consider top 95% probability mass
    min_tokens=20,
    max_tokens=100,
    presence_penalty=1.1,  # Reduce repetition
    frequency_penalty=1.1,  # Reduce repetition
    stop=["\n\n", "###"],  # Stop sequences
    ignore_eos=True,
    skip_special_tokens=True,
)

llm = LLM(
    model="facebook/opt-125m",
    gpu_memory_utilization=0.85,
    max_num_batched_tokens=8192,
    max_num_seqs=256,
    block_size=16,
)

- By default, vLLM will use sampling parameters recommended by model creator by applying the `generation_config.json` from the Hugging Face model repository if it exists.
- In most cases, this will provide you with the best results by default if SamplingParams is not specified.
- However, if vLLM's default sampling parameters are preferred, please set `generation_config="vllm"` when creating the LLM instance.

In [ ]:
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print("======================")
    print(f"Prompt: {prompt!r}, Generated text: {generated_text}")

- The `llm.generate()` method does not automatically apply the model's chat template to the input prompt.
- Therefore, if you are using an Instruct model or Chat model, you should manually apply the corresponding chat template to ensure the expected behavior.
- Alternatively, you can use the `llm.chat` method and pass a list of messages which have the same format as those passed to OpenAI's `client.chat.completions`

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-4")
messages_list = [
    [{"role": "user", "content": prompt}]
    for prompt in prompts
]
texts = tokenizer.apply_chat_template(
    messages_list,
    tokenize=False,
    add_generation_prompt=True,
)

for text, msg in zip(texts, messages_list):
    print("================================")
    print(f"==========plain message dict: {msg}")
    print(f"==========message dict after applying chat template: {text}")

In [ ]:
print("using llm.generate() ============================================================")
outputs = llm.generate(texts, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print("================================")
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")


print("using llm.chat() ============================================================")
outputs = llm.chat(messages_list, sampling_params, chat_template=tokenizer.chat_template)
for idx, output in enumerate(outputs):
    prompt = prompts[idx]
    generated_text = output.outputs[0].text
    print("================================")
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

### Online Inference using OpenAI-Compatible Server

### setting up the server

```bash
vllm serve Qwen/Qwen2.5-1.5B-Instruct
```

```bash
python -m vllm.entrypoints.openai.api_server \
    --model HuggingFaceTB/SmolLM2-360M-Instruct \
    --host 0.0.0.0 \
    --port 8000
```

To disable using model's default `generation_config.json`, please pass `--generation-config vllm` when launching the server.

This server can be queried in the same format as OpenAI API. For example, to list the models
```bash
curl http://localhost:8000/v1/models
```

### Inference using curl

```bash
curl http://localhost:8000/v1/completions \
    -H "Content-Type: application/json" \
    -d '{
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "prompt": "San Francisco is a",
        "max_tokens": 7,
        "temperature": 0
    }'
```

```bash
curl http://localhost:8000/v1/chat/completions \
    -H "Content-Type: application/json" \
    -d '{
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Who won the world series in 2020?"}
        ]
    }'
```

### Inference using python

In [ ]:
from huggingface_hub import InferenceClient
from openai import OpenAI

base_uri = "http://localhost:8000/v1"
model="Qwen/Qwen2.5-1.5B-Instruct"

#### Inference using Hugging Face’s InferenceClient

In [ ]:
client = InferenceClient(model=base_uri)

response = client.chat_completion(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print(response.choices[0].message.content)

#### Inference using the OpenAI client

In [ ]:
client = OpenAI(base_url=base_uri,api_key="")

completion = client.completions.create(
    model=model,
    prompt="San Francisco is a",
)
print("completion result:", completion)

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a story"},
    ],
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
)
print("chat result", response)